In [4]:
import os, sys, time, urllib.request

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pybullet"], check=True)
    RAW = "https://raw.githubusercontent.com/LxViRaL-teaching/medical-robotics-2026/main/week-3/"
    for f in ["ctrl_utils.py", "pendulum.urdf", "arm_2r.urdf"]:
        if not os.path.exists(f):
            urllib.request.urlretrieve(RAW + f, f)

import numpy as np
import pybullet as p
import ctrl_utils as ctl

DT = ctl.DT   # time step of the simulation, and of our controller: 1/240 s

ModuleNotFoundError: No module named 'ctrl_utils'

In [ ]:
ctl.connect()
pendulum = ctl.load_robot("pendulum.urdf")
motor = ctl.joint_index(pendulum, "motor_joint")
ball = ctl.link_index(pendulum, "ball_link")

In [ ]:
ctl.reset_joints(pendulum, [motor], [np.pi/2])
t = 0.0
duration = 6.0
time_log, angle_log, torque_log = [], [], []

p.enableJointForceTorqueSensor(pendulum,motor,1)
while t < duration:

    p.stepSimulation()
    q,qd,tau,_ = p.getJointState(pendulum, motor)

    t = t + DT
    time.sleep(DT)

    # Logging:
    time_log.append(t)
    angle_log.append(q)

ctl.plot_series(time_log, {"Joint Position": angle_log}, ylabel="angle [rad]")

In [ ]:

# Controller settings
q_desired = np.pi / 2    # target angle [rad]: arm horizontal
Kp = 50                 # proportional gain [N m per rad of error]
duration = 6.0           # how long to run [s]

# Start with the arm hanging down, at rest
ctl.reset_joints(pendulum, [motor], [0.0])

time_log, angle_log, torque_log = [], [], []
t = 0.0
while t < duration:
    # 1. SENSE: read the joint angle from the encoder
    q = p.getJointState(pendulum, motor)[0]

    # 2. CONTROL: compute the motor torque from the error
    error = q_desired - q
    torque = error * Kp

    # 3. ACT: send the torque to the motor, and let the simulated world move for one time step
    p.setJointMotorControl2(pendulum, motor, p.TORQUE_CONTROL, force=torque)

    
    # 4. WAIT: let one time step of real time pass, so what you see runs at real speed (comment out for faster simulation, or sleep for DT/4)
    p.stepSimulation()
    t = t + DT
    
    time.sleep(DT)

    # Logging:
    time_log.append(t)
    angle_log.append(q)
    torque_log.append(torque)

p_time, p_angle, p_torque = time_log, angle_log, torque_log   # keep this run, to compare later
ctl.plot_series(time_log, {"P": angle_log}, ylabel="angle [rad]", ref=q_desired)
ctl.plot_series(time_log, {"P": torque_log}, ylabel="torque [N m]")

